# W11｜聲音剪輯與訊號處理

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/willismax/MediaSystem-Python-Course/blob/main/course-115-v2/colab/W11_%E8%81%B2%E9%9F%B3%E5%89%AA%E8%BC%AF%E8%88%87%E8%A8%8A%E8%99%9F%E8%99%95%E7%90%86.ipynb)

- **對應講義**：[HackMD 第 11 章　聲音剪輯與訊號處理](https://hackmd.io/WYh6ttxqTa6imF-B_Dn_cg)
- **互動實驗室**：[W11 Lab（瀏覽器版）](https://willismax.github.io/MediaSystem-Python-Course/labs/w11/)
- **先備**：W10 取樣、波形、FFT、STFT／頻譜圖

本週把 W10「看懂聲音」延伸到「動手改聲音」：用 **pydub** 做裁切、增益、串接、混音、淡入淡出、靜音偵測與 Podcast 自動剪輯；再用 **librosa** 做變速不變調與變調不變速，並把結果拉回 W10 的頻譜觀念檢查。

| 學習目標 | 你要能做到 |
|---|---|
| 剪輯的單位 | 說明 pydub 切片以**毫秒**為單位，numpy 陣列以**樣本**為單位 |
| 增益與混音 | 分辨 `audio + 6`（增益）、`audio + audio2`（串接）、`overlay`（疊加），並避免削波 |
| 靜音偵測 | 用相對門檻（`audio.dBFS - 14`）找出語音段，壓縮過長停頓 |
| 音量一致 | 分辨峰值正規化、平均 dBFS 平移與 LUFS，知道長靜音為何會讓平均音量失真 |
| 時間與音高 | 說明 time stretch／pitch shift 和「直接改播放速度」的差別，以及它們與 STFT 的關係 |

**使用方式**：從上到下依序執行即可，**不需要上傳任何檔案**——示範素材會在程式中自動產生。想用自己的錄音，請看「0.4 選用：上傳自己的錄音」。

> 圖表的座標軸標籤一律使用英文（避免 Colab 缺中文字型而出現方框），說明文字則用中文。

## 本週流程地圖

| 段落 | 主題 | 主要程式 |
|---|---|---|
| 0 | 環境、工具函式、示範素材 | `np_to_seg()`、`seg_to_np()`、`play()` |
| 11.5 | 讀取音訊與 metadata | `AudioSegment.from_file()` |
| 11.6 | 切片以毫秒計 | `audio[5000:15000]` |
| 11.7 | 增益、串接、疊加、削波 | `clip + 6`、`clip + louder`、`voice.overlay(bgm)` |
| 11.8 | 淡入淡出與交叉淡化 | `fade_in()`、`fade_out()`、`append(crossfade=)` |
| 練習 1 | 裁切＋淡入淡出＋增益 | |
| 11.9–11.12 | 靜音偵測、相對門檻、邊界情況 | `detect_nonsilent()`、`get_relative_threshold()` |
| 11.14 | 壓縮過長停頓 | `compact_long_silence()` |
| 11.15 | 三種「正規化」 | `shift_average_dbfs()` |
| 11.16–11.17、11.27 | 片頭片尾、完整 Podcast Editor、處理紀錄 | `podcast_editor()` |
| 練習 2、3 | Podcast 自動剪輯、參數掃描 | |
| 11.18–11.19 | 背景音樂與 ducking | `overlay(loop=True)` |
| 11.20 | 高通／低通濾波與頻譜 | `high_pass_filter()`、`low_pass_filter()` |
| 11.21–11.22 | 變速不變調、變調不變速 | `librosa.effects.time_stretch()`、`pitch_shift()` |
| 延伸 | 參數比較 | |
| 檢核 | 檢核題、繳交清單 | |

## 0.1 安裝套件

- `pydub`：以「毫秒」為單位的剪輯工具，底層呼叫 `audioop` 與 `ffmpeg`。
- `audioop` 從 **Python 3.13** 起被移出標準函式庫，所以用 `audioop-lts` 補回來（較舊的 Python 會自動略過）。
- Colab 已內建 `ffmpeg`，匯出 MP3 不需另外安裝。

In [ ]:
!pip -q install pydub librosa soundfile "audioop-lts; python_version>='3.13'"

## 0.2 匯入與工具函式

pydub 的 `AudioSegment` 和 librosa 的 numpy 陣列是兩種不同的資料表示：

| | pydub `AudioSegment` | librosa / numpy |
|---|---|---|
| 內容 | 整數 PCM bytes＋metadata | 浮點數陣列（約 −1～1） |
| 切片單位 | **毫秒** | **樣本** |
| 擅長 | 剪接、增益、淡入淡出、匯出 | 頻譜分析、STFT、變速變調 |

下面的 `np_to_seg()`、`seg_to_np()` 負責兩者互轉。`play()` 把聲音先存成 WAV 再播放，**刻意不用** `Audio(y, rate=sr)`，因為那種寫法預設會自動把音量正規化，你就聽不出增益前後的差別了。

In [ ]:
import io
import warnings
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import librosa
import librosa.display
import soundfile as sf
from pydub import AudioSegment, effects
from pydub.silence import detect_nonsilent
from IPython.display import Audio, Markdown, display

SR = 44100                      # 本週統一使用 44.1 kHz（CD／Podcast 常用取樣率）
warnings.filterwarnings("ignore", message="invalid value encountered in cast")  # librosa/numba 的無害警告
rng = np.random.default_rng(115)
plt.rcParams.update({"figure.figsize": (11, 3), "axes.grid": True, "grid.alpha": 0.3})


def np_to_seg(y, sr=SR):
    """numpy 浮點陣列（-1~1，形狀 (n,) 或 (n, 聲道)）→ 16-bit AudioSegment。"""
    y = np.asarray(y, dtype=np.float32)
    channels = 1 if y.ndim == 1 else y.shape[1]
    pcm = (np.clip(y.reshape(-1), -1.0, 1.0) * 32767).astype(np.int16)
    return AudioSegment(data=pcm.tobytes(), sample_width=2, frame_rate=sr, channels=channels)


def seg_to_np(seg, mono=True):
    """AudioSegment → (numpy 浮點陣列, 取樣率)。多聲道預設平均成單聲道。"""
    samples = np.array(seg.get_array_of_samples(), dtype=np.float32)
    y = samples / float(1 << (8 * seg.sample_width - 1))
    if seg.channels > 1:
        y = y.reshape(-1, seg.channels)
        if mono:
            y = y.mean(axis=1)
    return y, seg.frame_rate


def play(seg, label=""):
    """以 WAV 原樣播放（不自動正規化音量）。"""
    if label:
        display(Markdown(f"**聆聽：{label}**"))
    buf = io.BytesIO()
    seg.export(buf, format="wav")
    display(Audio(data=buf.getvalue()))


def count_clipped(seg):
    """計算貼到滿刻度（可能削波）的樣本數。"""
    samples = np.abs(np.array(seg.get_array_of_samples(), dtype=np.int64))
    return int(np.sum(samples >= seg.max_possible_amplitude - 1))


def describe(seg, name):
    print(f"{name:<18} 長度 {len(seg) / 1000:7.2f} s | dBFS {seg.dBFS:7.2f} | "
          f"max_dBFS {seg.max_dBFS:6.2f} | 滿刻度樣本 {count_clipped(seg)}")


def plot_waves(segs, titles, ylim=1.05):
    fig, axes = plt.subplots(len(segs), 1, figsize=(11, 2.2 * len(segs)), squeeze=False)
    for ax, seg, title in zip(axes[:, 0], segs, titles):
        y, sr = seg_to_np(seg)
        ax.plot(np.arange(len(y)) / sr, y, lw=0.4)
        ax.set_ylim(-ylim, ylim)
        ax.set_title(title)
        ax.set_ylabel("Amplitude")
    axes[-1, 0].set_xlabel("Time (s)")
    plt.tight_layout()
    plt.show()


def plot_spans(seg, spans, title):
    """畫波形並把 detect_nonsilent 找到的語音段塗上顏色。"""
    y, sr = seg_to_np(seg)
    plt.figure(figsize=(11, 2.6))
    plt.plot(np.arange(len(y)) / sr, y, lw=0.4, color="0.3")
    for start, end in spans:
        plt.axvspan(start / 1000, end / 1000, color="tab:orange", alpha=0.3)
    plt.ylim(-1.05, 1.05)
    plt.title(f"{title}  ({len(spans)} spans)")
    plt.xlabel("Time (s)")
    plt.ylabel("Amplitude")
    plt.tight_layout()
    plt.show()


def avg_spectrum_db(seg, n_fft=4096):
    """把 STFT 各影格的幅度取平均 → 平均頻譜（dB）。W10 STFT 的延伸用法。"""
    y, sr = seg_to_np(seg)
    S = np.abs(librosa.stft(y, n_fft=n_fft, hop_length=n_fft // 4))
    return librosa.fft_frequencies(sr=sr, n_fft=n_fft), 20 * np.log10(S.mean(axis=1) + 1e-10)


def plot_spectra(segs, labels, title):
    plt.figure(figsize=(11, 3.6))
    for seg, label in zip(segs, labels):
        f, db = avg_spectrum_db(seg)
        plt.semilogx(f[1:], db[1:], lw=1, label=label)
    plt.xlim(30, SR / 2)
    plt.xlabel("Frequency (Hz, log)")
    plt.ylabel("Mean magnitude (dB)")
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.show()


def show_table(header, rows):
    """以 Markdown 表格顯示結果。"""
    lines = ["| " + " | ".join(header) + " |", "|" + "---|" * len(header)]
    for r in rows:
        lines.append("| " + " | ".join(f"{v:.2f}" if isinstance(v, float) else str(v) for v in r) + " |")
    display(Markdown("\n".join(lines)))


print("工具函式就緒")

## 0.3 產生示範素材（不需上傳）

這一格會建立四個檔案：

| 檔案 | 內容 |
|---|---|
| `recording.wav` | 語音。優先下載 `librosa.ex('libri1')`（LibriSpeech 真實朗讀）；下載失敗就改用程式合成的「類語音」音節串。程式會把語句拆開，**刻意插入 1.5–2.5 秒的長停頓**（以及少數 0.3–0.5 秒的短停頓），並加上約 −62 dBFS 的底噪，模擬真實錄音。 |
| `bgm.wav` | 8 秒可循環的背景音樂（C–Am–F–G 和弦鋪底＋琶音）|
| `intro.wav`／`outro.wav` | 片頭、片尾短音效 |

> **先預測再執行**：插入很多長停頓之後，整段錄音的平均音量（dBFS）會變高還是變低？

In [ ]:
def room_tone(n_samples, level_db=-62):
    """模擬錄音室底噪：很小的白雜訊。"""
    return (rng.standard_normal(n_samples) * 10 ** (level_db / 20)).astype(np.float32)


def load_libri_phrases():
    """下載 librosa 範例語音，依停頓切成數個語句（每句最長約 4 秒）。"""
    y, _ = librosa.load(librosa.ex("libri1"), sr=SR, mono=True)
    intervals = librosa.effects.split(y, top_db=35)
    phrases, cur = [], list(intervals[0])
    for s, e in intervals[1:]:
        if s - cur[1] < 0.35 * SR and e - cur[0] < 4.0 * SR:
            cur[1] = e
        else:
            phrases.append(cur)
            cur = [s, e]
    phrases.append(cur)
    pad = int(0.05 * SR)
    return [y[max(0, s - pad): e + pad] for s, e in phrases if e - s > 0.3 * SR]


VOWELS = [(730, 1090), (270, 2290), (300, 870), (530, 1840), (570, 840), (440, 1020)]


def synth_syllable(dur, f0):
    """一個「類母音」音節：諧波 × 共振峰權重 × 包絡，偶爾帶一個子音雜訊。"""
    n = int(dur * SR)
    t = np.arange(n) / SR
    f_inst = f0 * (1 + 0.06 * np.sin(2 * np.pi * rng.uniform(1, 3) * t))
    phase = 2 * np.pi * np.cumsum(f_inst) / SR
    F1, F2 = VOWELS[rng.integers(len(VOWELS))]
    sig = np.zeros(n)
    for k in range(1, 60):
        fk = f0 * k
        if fk > 5000:
            break
        w = np.exp(-((fk - F1) / 200) ** 2) + 0.6 * np.exp(-((fk - F2) / 300) ** 2) + 0.03 / k
        sig += w * np.sin(k * phase)
    sig *= np.sin(np.pi * np.linspace(0, 1, n)) ** 0.7
    if rng.random() < 0.5:                                   # 子音：短促的高頻雜訊
        burst = np.diff(rng.standard_normal(int(0.04 * SR) + 1)) * 0.15
        sig = np.concatenate([burst * np.hanning(len(burst)), sig])
    return sig


def synth_phrase():
    n_syl = int(rng.integers(6, 13))
    f0_start = rng.uniform(120, 180)
    parts = []
    for i in range(n_syl):
        f0 = f0_start * (1 - 0.25 * i / n_syl)               # 句尾音高下降（語調）
        parts.append(synth_syllable(rng.uniform(0.12, 0.28), f0))
        parts.append(room_tone(int(rng.uniform(0.02, 0.08) * SR)))
    p = np.concatenate(parts)
    return p / np.abs(p).max() * rng.uniform(0.7, 1.0)


try:
    phrases = load_libri_phrases()
    SOURCE = "librosa.ex('libri1')：真實語音"
except Exception as err:                                     # 無網路、被防火牆擋等
    print(f"範例語音下載失敗（{type(err).__name__}），改用合成語音。")
    phrases = [synth_phrase() for _ in range(9)]
    SOURCE = "合成的類語音音節"

peak = max(np.abs(p).max() for p in phrases)
phrases = [p * (0.5 / peak) for p in phrases]                # 語音峰值約 -6 dBFS
if sum(len(p) for p in phrases) < 12 * SR:                   # 太短就重複一輪，確保總長 > 20 秒
    phrases = phrases * 2

pieces = [room_tone(int(1.0 * SR))]
for i, p in enumerate(phrases):
    pieces.append(p + room_tone(len(p)))
    if i < len(phrases) - 1:
        pause = rng.uniform(0.3, 0.5) if i % 3 == 2 else rng.uniform(1.5, 2.5)
        pieces.append(room_tone(int(pause * SR)))
pieces.append(room_tone(int(1.5 * SR)))
speech = np.concatenate(pieces).astype(np.float32)
sf.write("recording.wav", speech, SR, subtype="PCM_16")


# ---- 背景音樂：C–Am–F–G，每個和弦 2 秒，共 8 秒可循環 ----
def midi_hz(m):
    return 440.0 * 2 ** ((m - 69) / 12)


def tone(freq, dur, decay=0.0, harmonics=(1.0, 0.3, 0.1)):
    t = np.arange(int(dur * SR)) / SR
    sig = sum(a * np.sin(2 * np.pi * freq * (k + 1) * t) for k, a in enumerate(harmonics))
    env = np.exp(-decay * t) if decay else np.ones_like(t)
    fade = np.minimum(1, np.minimum(t / 0.01, (dur - t) / 0.02))   # 避免爆音
    return sig * env * fade


CHORDS = [[60, 64, 67], [57, 60, 64], [53, 57, 60], [55, 59, 62]]
bgm_parts = []
for chord in CHORDS:
    seg = sum(0.10 * tone(midi_hz(m), 2.0) for m in chord)        # 和弦鋪底
    seg += 0.25 * tone(midi_hz(chord[0] - 24), 2.0, decay=1.5)     # 低音
    arp = np.concatenate([0.22 * tone(midi_hz(chord[i % 3] + 12), 0.25, decay=9)
                          for i in range(8)])                      # 八分音符琶音
    bgm_parts.append(seg + arp[: len(seg)])
bgm_np = np.concatenate(bgm_parts)
bgm_np = bgm_np / np.abs(bgm_np).max() * 0.6
sf.write("bgm.wav", bgm_np.astype(np.float32), SR, subtype="PCM_16")

# ---- 片頭：上行琶音後停在 C 大三和弦；片尾：F → C 終止 ----
intro_np = np.zeros(int(3.0 * SR))
for i, m in enumerate([60, 64, 67, 72, 76]):
    start = int(i * 0.25 * SR)
    note = 0.3 * tone(midi_hz(m), 3.0 - i * 0.25, decay=1.2)
    intro_np[start:start + len(note)] += note
outro_np = np.concatenate([sum(0.2 * tone(midi_hz(m), 1.5, decay=0.8) for m in [53, 57, 60, 65]),
                           sum(0.2 * tone(midi_hz(m), 2.5, decay=0.6) for m in [48, 55, 60, 64])])
for name, sig in [("intro.wav", intro_np), ("outro.wav", outro_np)]:
    sf.write(name, (sig / np.abs(sig).max() * 0.6).astype(np.float32), SR, subtype="PCM_16")

print("語音來源：", SOURCE)
for name in ["recording.wav", "bgm.wav", "intro.wav", "outro.wav"]:
    info = sf.info(name)
    print(f"{name:<14} {info.duration:6.2f} s  {info.samplerate} Hz  {info.channels} ch")

In [ ]:
for name in ["recording.wav", "bgm.wav", "intro.wav", "outro.wav"]:
    play(AudioSegment.from_file(name), name)
plot_waves([AudioSegment.from_file("recording.wav")], ["recording.wav (with inserted long pauses)"])

### 0.4 選用：上傳自己的錄音

想把本週所有流程套用到自己的錄音（建議 1–3 分鐘的口說內容）：

1. 把下一格的 `USE_MY_RECORDING` 改成 `True` 再執行，選擇檔案（wav／mp3／m4a 皆可）。
2. 檔案會被轉成單聲道、44.1 kHz，覆蓋 `recording.wav`。
3. 從 **11.5** 開始重新往下執行。

預設為 `False`，所以「全部執行」時不會停下來等你上傳。

In [ ]:
USE_MY_RECORDING = False   # 改成 True 才會跳出上傳視窗

if USE_MY_RECORDING:
    try:
        from google.colab import files
        uploaded = files.upload()
        if uploaded:
            name = next(iter(uploaded))
            mine = AudioSegment.from_file(name).set_channels(1).set_frame_rate(SR).set_sample_width(2)
            mine.export("recording.wav", format="wav")
            print(f"已用 {name} 取代 recording.wav（{len(mine) / 1000:.1f} 秒）")
        else:
            print("沒有選擇檔案，沿用示範素材。")
    except Exception as err:
        print(f"略過上傳（{type(err).__name__}: {err}），沿用示範素材。")
else:
    print("使用示範素材 recording.wav。")

## 11.5 讀取音訊與 metadata

**概念**：剪輯前先「看規格」。`len(audio)` 是**毫秒**；`frame_rate` 就是 W10 的取樣率；`sample_width` 是每個樣本幾個 byte（2 = 16-bit）；`dBFS` 是整段的平均音量（RMS），`max_dBFS` 是峰值，兩者都以「滿刻度 = 0 dB」為基準，所以一定 ≤ 0。

> **先預測再執行**：`dBFS` 和 `max_dBFS` 哪一個比較接近 0？差多少算正常？

In [ ]:
audio = AudioSegment.from_file("recording.wav")

print("長度（毫秒）:", len(audio))
print("聲道數:", audio.channels)
print("取樣率（Hz）:", audio.frame_rate)
print("取樣寬度（bytes）:", audio.sample_width)
print("平均音量 dBFS:", round(audio.dBFS, 2))
print("峰值 max_dBFS:", round(audio.max_dBFS, 2))

# 連回 W10：樣本數 = 取樣率 × 秒數
print("樣本數 frame_count():", int(audio.frame_count()),
      "≈ 取樣率 × 秒數 =", audio.frame_rate * len(audio) / 1000)

**觀察**：峰值與平均值通常差 15–25 dB。差距越大，代表聲音「動態」越大，或有越多安靜的段落。

**聆聽檢核**
- [ ] 我能說出這段錄音的取樣率、聲道數、時長。
- [ ] 我知道 `dBFS` 是平均、`max_dBFS` 是峰值。

## 11.6 切片：單位是毫秒

**概念**：pydub 的切片 `audio[a:b]` 以**毫秒**計算，不是秒、也不是樣本。這是本週最常見的錯誤。

> **先預測再執行**：`audio[5:15]` 會得到幾秒的聲音？聽起來像什麼？

In [ ]:
clip = audio[5000:15000]   # 第 5 秒到第 15 秒
trap = audio[5:15]         # 陷阱：這是第 5 毫秒到第 15 毫秒！

print("clip 長度:", len(clip), "ms")
print("trap 長度:", len(trap), "ms  ← 只有 10 毫秒，不是 10 秒")

# 對照 numpy：陣列索引是「樣本」，要乘上取樣率
y, sr = seg_to_np(audio)
y_clip = y[5 * sr: 15 * sr]
print("numpy 切片:", len(y_clip), "個樣本 =", len(y_clip) / sr, "秒")

play(clip, "clip = audio[5000:15000]（10 秒）")
play(trap, "trap = audio[5:15]（10 毫秒，幾乎只有一聲「喀」）")

**聆聽檢核**
- [ ] `clip` 約 10 秒；`trap` 短到幾乎聽不到。
- [ ] 我能寫出「pydub 用毫秒、numpy 用樣本」的換算：樣本 = 毫秒 × 取樣率 ÷ 1000。

## 11.7 增益、串接與疊加

**概念**：同樣是 `+`，右邊放的東西不同，意義完全不同：

| 寫法 | 意義 | 等價寫法 | 長度 |
|---|---|---|---|
| `clip + 6` | 增益 +6 dB | `clip.apply_gain(6)` | 不變 |
| `clip + louder` | 串接（接在後面）| `clip.append(louder, crossfade=0)` | 相加 |
| `voice.overlay(bgm)` | 疊加（同時播放、混音）| — | 以 `voice` 為準 |

**增益公式**：振幅 × 10^(dB/20)。所以 +6 dB ≈ ×2、−6 dB ≈ ×0.5、+20 dB = ×10。

> **先預測再執行**：`clip + 6` 之後，RMS 會變成原來的幾倍？`max_dBFS` 會變成多少？

In [ ]:
for db in [-20, -12, -6, -3, 0, 3, 6, 12, 20]:
    print(f"{db:+4d} dB → 振幅 × {10 ** (db / 20):6.3f}")

In [ ]:
louder = clip + 6
same = clip.apply_gain(6)
print("clip + 6 與 apply_gain(6) 結果相同:", louder.raw_data == same.raw_data)
print(f"RMS 倍數: {louder.rms / clip.rms:.3f}（理論值 10^(6/20) = {10 ** 0.3:.3f}）")
describe(clip, "clip")
describe(louder, "clip + 6")

combined = clip + louder
print("\n串接長度:", len(clip), "+", len(louder), "=", len(combined), "ms")
print("與 append(crossfade=0) 相同:", combined.raw_data == clip.append(louder, crossfade=0).raw_data)

bgm = AudioSegment.from_file("bgm.wav")
voice = clip
mix = voice.overlay(bgm)
print("\n疊加長度: voice", len(voice), "ms，bgm", len(bgm), "ms → overlay", len(mix), "ms")

play(louder, "clip + 6（增益）")
play(combined, "clip + louder（串接：先原音量、再大聲）")
play(mix, "voice.overlay(bgm)（疊加）")

**觀察**：`overlay` 的長度以呼叫者 `voice` 為準；`bgm` 只有 8 秒，所以最後 2 秒沒有音樂（11.18 會用 `loop=True` 解決）。

### 削波（clipping）與餘裕（headroom）

兩個訊號疊加，振幅會**相加**。16-bit 整數最多只能表示 ±32767，超過的部分會被「削平」，聽起來是刺耳的破音。解法是在疊加**之前**先把各軌調小，留下餘裕。

> **先預測再執行**：`(clip + 6).overlay(bgm + 3)` 的 `max_dBFS` 會是多少？會有多少滿刻度樣本？

In [ ]:
hot_mix = (clip + 6).overlay(bgm + 3)        # 兩軌都很大聲直接疊
safe_mix = (clip - 3).overlay(bgm - 12)      # 先降音量再疊，留 headroom

describe(hot_mix, "hot_mix")
describe(safe_mix, "safe_mix")

# 用浮點數相加看「理論上」超出多少
yv, _ = seg_to_np(clip + 6)
yb, _ = seg_to_np(bgm + 3)
n = min(len(yv), len(yb))
print(f"\n浮點相加的峰值: {np.abs(yv[:n] + yb[:n]).max():.2f}（> 1.0 的部分在 16-bit 中會被削掉）")

plot_waves([hot_mix, safe_mix], ["hot_mix: (clip+6).overlay(bgm+3) - clipped", "safe_mix: (clip-3).overlay(bgm-12)"])
play(hot_mix, "hot_mix（注意破音）")
play(safe_mix, "safe_mix")

**聆聽檢核**
- [ ] `clip + 6` 明顯比較大聲，但長度不變。
- [ ] `hot_mix` 的波形頂部被削平，聽得出破音；`safe_mix` 沒有。
- [ ] 我能用一句話說明 headroom：先留空間，再疊加。

## 11.8 淡入、淡出與交叉淡化

**概念**：直接切斷聲音，波形會從非零值瞬間跳到 0，產生「喀」聲。淡入淡出就是在開頭／結尾乘上一段漸變的增益曲線。交叉淡化（crossfade）讓 A 的結尾淡出、B 的開頭淡入，兩者**重疊**一段時間。

> **先預測再執行**：`part_a.append(part_b, crossfade=1000)` 的總長度等於 `len(part_a) + len(part_b)` 嗎？

In [ ]:
faded = clip.fade_in(1000).fade_out(2000)

part_a = audio[5000:10000]
part_b = bgm[:5000] - 6
joined = part_a.append(part_b, crossfade=1000)
hard = part_a + part_b

print("part_a:", len(part_a), "ms | part_b:", len(part_b), "ms")
print("直接串接:", len(hard), "ms | crossfade=1000:", len(joined), "ms（重疊 1000 ms）")

plot_waves([clip, faded, hard, joined],
           ["clip", "clip.fade_in(1000).fade_out(2000)",
            "part_a + part_b (hard cut)", "part_a.append(part_b, crossfade=1000)"])
play(faded, "淡入 1 秒、淡出 2 秒")
play(hard, "硬接")
play(joined, "交叉淡化 1 秒")

**聆聽檢核**
- [ ] 淡入淡出的波形包絡呈現「斜坡」。
- [ ] 交叉淡化的總長度少了 1000 ms，轉場比硬接自然。

## 練習 1：裁切＋淡入淡出＋增益

1. 修改 `TODO` 參數：挑一段 8–12 秒、包含完整語句的區間。
2. 執行後比較前後波形，並把下表填完（數字可直接從輸出複製）。

| 項目 | 處理前 | 處理後 |
|---|---|---|
| 起訖（ms）| | — |
| 長度（s）| | |
| dBFS | | |
| max dBFS | | |
| 滿刻度樣本數 | | |
| 淡入／淡出（ms）、增益（dB）| — | |
| 一句聆聽描述 | | |

In [ ]:
# TODO 1：改成你要保留的起訖時間（毫秒）
START_MS, END_MS = 2000, 12000
# TODO 2：設定淡入、淡出長度（毫秒）與增益（dB）
FADE_IN_MS, FADE_OUT_MS, GAIN_DB = 500, 1500, 3

ex1_before = audio[START_MS:END_MS]
ex1_after = ex1_before.fade_in(FADE_IN_MS).fade_out(FADE_OUT_MS) + GAIN_DB

describe(ex1_before, "before")
describe(ex1_after, "after")
plot_waves([ex1_before, ex1_after], ["Exercise 1 - before", "Exercise 1 - after (fade + gain)"])
play(ex1_before, "處理前")
play(ex1_after, "處理後")

## 11.9 靜音偵測：`detect_nonsilent`

**概念**：pydub 以 1 ms 為步進，檢查每個長度為 `min_silence_len` 的視窗，RMS 低於 `silence_thresh`（dBFS）就算靜音；`detect_nonsilent` 回傳靜音以外的區段 `[start_ms, end_ms]`。

- `min_silence_len=800`：至少安靜 800 ms 才算「停頓」，字與字之間的短空隙不會被切開。
- `silence_thresh=-40`：低於 −40 dBFS 視為靜音（**絕對門檻**）。

> **先預測再執行**：示範錄音中插入了幾個長停頓？會偵測到幾段語音？

In [ ]:
spans = detect_nonsilent(audio, min_silence_len=800, silence_thresh=-40)
print("偵測到", len(spans), "段語音（毫秒）:")
for start, end in spans:
    print(f"  {start:6d} – {end:6d}  ({(end - start) / 1000:.2f} s)")
plot_spans(audio, spans, "detect_nonsilent(min_silence_len=800, silence_thresh=-40)")

**觀察**：0.3–0.5 秒的短停頓短於 800 ms，所以前後兩句會被併成同一段。

## 11.10 相對門檻：`audio.dBFS - 14`

**概念**：−40 dBFS 對大聲的錄音剛好，對小聲的錄音卻可能把語音也當成靜音。改用「比整段平均低 14 dB」當門檻，門檻會跟著錄音音量走。

> **先預測再執行**：把錄音整體調小 12 dB（模擬麥克風離太遠），絕對門檻 −40 與相對門檻各會找到幾段？

In [ ]:
threshold = audio.dBFS - 14
quiet = audio - 12

cases = [
    ("原錄音", audio, -40),
    ("原錄音", audio, threshold),
    ("小聲 12 dB", quiet, -40),
    ("小聲 12 dB", quiet, quiet.dBFS - 14),
]
rows = []
for name, seg, thr in cases:
    sp = detect_nonsilent(seg, min_silence_len=800, silence_thresh=thr)
    kind = "絕對 -40" if thr == -40 else "相對 dBFS-14"
    rows.append((name, kind, round(thr, 2), len(sp), round(sum(e - s for s, e in sp) / 1000, 2)))
show_table(["錄音", "門檻類型", "silence_thresh", "語音段數", "語音總長 (s)"], rows)

plot_spans(quiet, detect_nonsilent(quiet, 800, -40), "quiet recording, absolute -40 dBFS")
plot_spans(quiet, detect_nonsilent(quiet, 800, quiet.dBFS - 14), "quiet recording, relative dBFS-14")

**觀察**：相對門檻在兩種音量下的結果幾乎相同；絕對門檻遇到小聲錄音時，語音總長（有時連段數）會改變——較弱的音節被當成靜音。

## 11.12 邊界情況：全靜音的 dBFS 是 −∞

**概念**：dBFS = 20·log10(RMS ÷ 滿刻度)。全靜音的 RMS = 0，而 log10(0) = −∞。拿 −∞ 去算門檻或增益，後面全部壞掉，所以要先檢查。

> **先預測再執行**：`AudioSegment.silent(duration=5000).dBFS` 會印出什麼？

In [ ]:
silent = AudioSegment.silent(duration=5000)
print("全靜音 dBFS:", silent.dBFS, "| rms:", silent.rms)


def get_relative_threshold(audio, offset_db=14):
    """回傳相對靜音門檻；空音訊或全靜音回傳 None。"""
    if len(audio) == 0 or audio.rms == 0:
        return None
    return audio.dBFS - offset_db


print("全靜音:", get_relative_threshold(silent))
print("空音訊:", get_relative_threshold(AudioSegment.empty()))
print("recording.wav:", round(get_relative_threshold(audio), 2))

## 11.14 壓縮過長停頓：`compact_long_silence`

**概念**：找出語音段，把它們之間的長停頓**換成固定長度**（`keep_gap_ms`）的短靜音，而不是全部刪掉。

流程：相對門檻 → `detect_nonsilent` → 語音段之間插入 `AudioSegment.silent(duration=keep_gap_ms, frame_rate=audio.frame_rate)` → 串接。全靜音或找不到語音段時，原樣回傳。

> **先預測再執行**：處理後的長度大約剩多少秒？`dBFS` 會上升還是下降？

In [ ]:
def compact_long_silence(audio, min_silence_len=800, relative_thresh_db=14, keep_gap_ms=300):
    """把超過 min_silence_len 的停頓壓縮成 keep_gap_ms。"""
    if len(audio) == 0 or audio.rms == 0:          # 空音訊或全靜音：dBFS 是 -inf，無法算門檻
        print("compact_long_silence：輸入為空或全靜音，原樣回傳。")
        return audio
    silence_thresh = audio.dBFS - relative_thresh_db
    spans = detect_nonsilent(audio, min_silence_len=min_silence_len, silence_thresh=silence_thresh)
    if not spans:
        print("compact_long_silence：找不到語音段，原樣回傳。")
        return audio
    gap = AudioSegment.silent(duration=keep_gap_ms, frame_rate=audio.frame_rate)
    result = audio[spans[0][0]:spans[0][1]]
    for start, end in spans[1:]:
        result = result + gap + audio[start:end]
    return result


compact = compact_long_silence(audio, min_silence_len=800, relative_thresh_db=14, keep_gap_ms=300)
no_gap = compact_long_silence(audio, min_silence_len=800, relative_thresh_db=14, keep_gap_ms=0)

describe(audio, "原錄音")
describe(compact, "壓縮停頓 300 ms")
describe(no_gap, "停頓全刪 0 ms")
plot_waves([audio, compact, no_gap], ["original", "compact_long_silence(keep_gap_ms=300)", "keep_gap_ms=0"])
play(audio, "原錄音")
play(compact, "壓縮停頓（保留 300 ms）")
play(no_gap, "停頓全刪（keep_gap_ms=0）")

**觀察**：`compact` 的 dBFS 比原錄音高，但我們**沒有**調任何增益——只是靜音變少，平均值就變了（11.15 會用到這件事）。

**聆聽檢核**
- [ ] 壓縮後節奏變緊湊，但句子之間仍有呼吸感。
- [ ] `keep_gap_ms=0` 聽起來句子黏在一起、很急促——這就是「不能把停頓全刪」的原因。
- [ ] 句首或句尾有沒有被切掉一點點？（若有，可試著降低 `relative_thresh_db` 或在語音段前後多留幾十毫秒。）

## 11.15 三種「正規化」

日常說的「正規化」至少有三種，目標不同：

| 方法 | 對齊什麼 | 程式 | 優點 | 限制 |
|---|---|---|---|---|
| 峰值正規化 | 最大峰值（例如 −1 dBFS）| `effects.normalize(seg, headroom=1.0)` | 保證不削波 | 一個瞬間尖峰就決定整段音量 |
| 平均 dBFS 平移 | 平均 RMS（例如 −18 dBFS）| `shift_average_dbfs(seg, -18)` | 簡單、多集音量較一致 | 受靜音比例影響；可能把峰值推爆 |
| LUFS 響度 | 人耳感知響度（例如 −16 LUFS）| `pyloudnorm`（選用）| 平台標準（Podcast、串流）| 需額外套件 |

> **先預測再執行**：`compact` 平均約 −2x dBFS、峰值約 −6 dBFS。平移到 −18 dBFS 後，峰值會不會超過 0？

In [ ]:
def shift_average_dbfs(audio, target_dbfs=-18.0):
    """把整段平均音量平移到 target_dbfs。全靜音（-inf）原樣回傳。"""
    if audio.dBFS == float("-inf"):
        return audio
    gain_db = target_dbfs - audio.dBFS
    return audio.apply_gain(gain_db)


peak_norm = effects.normalize(compact, headroom=1.0)
avg_shift = shift_average_dbfs(compact, -18.0)

describe(compact, "compact")
describe(peak_norm, "峰值正規化 -1")
describe(avg_shift, "平均平移 -18")
if avg_shift.max_dBFS > -1.0:
    print("警告：平移後峰值已接近或超過 0 dBFS，可能削波，需要降低 target 或加限制器。")

play(peak_norm, "峰值正規化")
play(avg_shift, "平均 dBFS 平移到 -18")

### 選用：LUFS（需要 `pyloudnorm`）

沒安裝時這格只會印出提示，不會報錯。要試的話，先執行 `!pip -q install pyloudnorm` 再重跑這格。

In [ ]:
try:
    import pyloudnorm as pyln
except ImportError:
    pyln = None
    print("（選用）尚未安裝 pyloudnorm；執行 !pip -q install pyloudnorm 後重跑本格即可體驗 LUFS。")

if pyln is not None:
    y_c, sr_c = seg_to_np(compact)
    meter = pyln.Meter(sr_c)
    lufs_before = meter.integrated_loudness(y_c)
    y_lufs = pyln.normalize.loudness(y_c, lufs_before, -16.0)
    lufs_seg = np_to_seg(y_lufs, sr_c)
    print(f"整合響度: {lufs_before:.2f} LUFS → {meter.integrated_loudness(y_lufs):.2f} LUFS")
    describe(lufs_seg, "LUFS -16")
    play(lufs_seg, "LUFS 正規化到 -16")

### 為什麼長靜音會讓平均音量失真？

dBFS 是「整段」的 RMS：靜音越多，平均越低。若在**壓縮停頓之前**就把平均調到 −18，程式會以為錄音很小聲而加太多增益；之後再刪掉靜音，剩下的語音就比 −18 大聲很多，峰值也更容易削波。

> **先預測再執行**：「先調音量、後壓縮停頓」與「先壓縮停頓、後調音量」，最後的 dBFS 各是多少？

In [ ]:
wrong_order = compact_long_silence(shift_average_dbfs(audio, -18.0))
right_order = shift_average_dbfs(compact_long_silence(audio), -18.0)

print(f"原錄音 dBFS {audio.dBFS:.2f}；壓縮後 dBFS {compact.dBFS:.2f}（只是少了靜音）")
describe(wrong_order, "先調音量再壓縮")
describe(right_order, "先壓縮再調音量")

**聆聽檢核**
- [ ] 我能說出三種正規化各自對齊的是什麼。
- [ ] 我能解釋為什麼 Podcast Editor 要「先壓縮停頓、再調整平均音量」。

## 11.16 片頭與片尾

**概念**：片頭片尾通常各自加淡入淡出，再和正文串接。`+` 兩邊都是 `AudioSegment`，所以是**串接**。

> **先預測再執行**：`final` 的長度 = 哪三個長度相加？

In [ ]:
intro = AudioSegment.from_file("intro.wav")
outro = AudioSegment.from_file("outro.wav")
processed = shift_average_dbfs(compact_long_silence(audio), -18.0)

intro = intro.fade_in(500).fade_out(1000)
outro = outro.fade_in(1000).fade_out(2000)
final = intro + processed + outro

print(f"{len(intro)} + {len(processed)} + {len(outro)} = {len(final)} ms")
play(final, "片頭 + 正文 + 片尾")

## 11.17 完整 Podcast Editor（含 11.27 處理紀錄）

**概念**：把前面的步驟串成一條固定的處理管線，並在每一步記錄長度、dBFS、max dBFS（11.27），之後才能比較參數、寫報告。

```
recording.wav → compact_long_silence(800, 14, 300) → shift_average_dbfs(-18)
              → fade_in(300).fade_out(800) → podcast_result.mp3 (192 kbps)
```

> **先預測再執行**：MP3 是有損壓縮。重新讀回 MP3 後，長度會和匯出前完全一樣嗎？

In [ ]:
def log_levels(rows, stage, seg):
    """11.27：記錄每一步的長度與音量。"""
    rows.append((stage, round(len(seg) / 1000, 2), round(seg.dBFS, 2), round(seg.max_dBFS, 2)))


def podcast_editor(input_path="recording.wav", output_path="podcast_result.mp3",
                   min_silence_len=800, relative_thresh_db=14, keep_gap_ms=300, target_dbfs=-18.0):
    rows = []
    audio = AudioSegment.from_file(input_path)
    log_levels(rows, "1 原始", audio)

    edited = compact_long_silence(audio, min_silence_len=min_silence_len,
                                  relative_thresh_db=relative_thresh_db, keep_gap_ms=keep_gap_ms)
    log_levels(rows, "2 壓縮停頓", edited)

    edited = shift_average_dbfs(edited, target_dbfs=target_dbfs)
    log_levels(rows, "3 平均音量平移", edited)

    edited = edited.fade_in(300).fade_out(800)
    log_levels(rows, "4 淡入淡出", edited)

    edited.export(output_path, format="mp3", bitrate="192k")
    return edited, rows


podcast, log_rows = podcast_editor("recording.wav", "podcast_result.mp3")
show_table(["步驟", "長度 (s)", "dBFS", "max dBFS"], log_rows)

reloaded = AudioSegment.from_file("podcast_result.mp3")
print(f"MP3 檔案大小: {Path('podcast_result.mp3').stat().st_size / 1024:.1f} KB")
print(f"匯出前 {len(podcast)} ms，讀回 MP3 {len(reloaded)} ms（若差幾毫秒，是 MP3 以影格為單位編碼、編碼器補 padding 造成）")
play(audio, "處理前 recording.wav")
display(Markdown("**聆聽：處理後 podcast_result.mp3**"))
display(Audio(filename="podcast_result.mp3"))

**聆聽檢核**
- [ ] 處理後節奏更緊湊，音量更穩定。
- [ ] 開頭與結尾沒有突兀的「喀」聲。
- [ ] 紀錄表中每一步的長度與 dBFS 變化，我都能解釋原因。

## 練習 2：Podcast 自動剪輯

1. （建議）在 0.4 上傳自己 1–3 分鐘的口說錄音；沒有的話用示範素材。
2. 修改 `TODO` 參數並執行，把結果填入下表；至少寫 **3 條聆聽筆記**（例如：哪裡被切太緊、哪句開頭被吃掉、音量是否穩定）。

| 項目 | 數值 |
|---|---|
| 原始時長（s）| |
| 輸出時長（s）| |
| 相對門檻 relative_thresh_db／實際 silence_thresh（dBFS）| |
| min_silence_len（ms）| |
| keep_gap_ms（ms）| |
| dBFS（前 → 後）| |
| max dBFS（前 → 後）| |
| 聆聽筆記 1 | |
| 聆聽筆記 2 | |
| 聆聽筆記 3 | |

In [ ]:
# TODO：調整參數，觀察輸出時長與聽感
MY_MIN_SILENCE_LEN = 800
MY_RELATIVE_THRESH_DB = 14
MY_KEEP_GAP_MS = 300
MY_TARGET_DBFS = -18.0

my_input = AudioSegment.from_file("recording.wav")
print(f"實際 silence_thresh = {my_input.dBFS:.2f} - {MY_RELATIVE_THRESH_DB} = "
      f"{my_input.dBFS - MY_RELATIVE_THRESH_DB:.2f} dBFS")
my_podcast, my_rows = podcast_editor("recording.wav", "my_podcast.mp3",
                                     min_silence_len=MY_MIN_SILENCE_LEN,
                                     relative_thresh_db=MY_RELATIVE_THRESH_DB,
                                     keep_gap_ms=MY_KEEP_GAP_MS,
                                     target_dbfs=MY_TARGET_DBFS)
show_table(["步驟", "長度 (s)", "dBFS", "max dBFS"], my_rows)
display(Audio(filename="my_podcast.mp3"))

## 練習 3：掃描 `min_silence_len`

固定 `relative_thresh_db=14`、`keep_gap_ms=300`，只改 `min_silence_len`，觀察語音段數與輸出時長。

> **先預測再執行**：`min_silence_len` 越小，語音段數會變多還是變少？輸出時長呢？

執行後完成下表，並回答：哪個值聽起來最自然？為什麼？

| min_silence_len (ms) | 語音段數 | 輸出時長 (s) | 聽感（太碎／剛好／太鬆）|
|---|---|---|---|
| 300 | | | |
| 600 | | | |
| 900 | | | |
| 1200 | | | |

In [ ]:
SWEEP = [300, 600, 900, 1200]   # TODO（選做）：加入你想測的其他值
sweep_rows, sweep_out = [], {}
thr = get_relative_threshold(audio, 14)
for msl in SWEEP:
    sp = detect_nonsilent(audio, min_silence_len=msl, silence_thresh=thr)
    out = compact_long_silence(audio, min_silence_len=msl, relative_thresh_db=14, keep_gap_ms=300)
    sweep_out[msl] = out
    sweep_rows.append((msl, len(sp), round(len(out) / 1000, 2)))
show_table(["min_silence_len (ms)", "語音段數", "輸出時長 (s)"], sweep_rows)

fig, ax1 = plt.subplots(figsize=(7, 3))
ax1.plot([r[0] for r in sweep_rows], [r[2] for r in sweep_rows], "o-", label="output duration (s)")
ax1.set_xlabel("min_silence_len (ms)")
ax1.set_ylabel("Output duration (s)")
ax2 = ax1.twinx()
ax2.plot([r[0] for r in sweep_rows], [r[1] for r in sweep_rows], "s--", color="tab:orange")
ax2.set_ylabel("Number of spans", color="tab:orange")
plt.title("Sweep of min_silence_len")
plt.tight_layout()
plt.show()

for msl, out in sweep_out.items():
    play(out, f"min_silence_len = {msl} ms")

## 11.18 加入背景音樂

**概念**：先做一張和人聲一樣長的靜音「畫布」，把降低 18 dB 的 BGM 以 `loop=True` 鋪滿畫布，最後再疊上人聲。這樣輸出長度由人聲決定，BGM 也不會在中途斷掉。

> **先預測再執行**：`bgm - 18` 的振幅約是原本的幾倍？

In [ ]:
voice = podcast                     # 11.17 的處理結果
bgm = AudioSegment.from_file("bgm.wav")

canvas = AudioSegment.silent(duration=len(voice), frame_rate=voice.frame_rate)
canvas = canvas.overlay(bgm - 18, loop=True)
mixed = canvas.overlay(voice)

describe(voice, "voice")
describe(canvas, "BGM 畫布 (-18 dB)")
describe(mixed, "mixed")
play(mixed, "人聲 + BGM（-18 dB）")

## 11.19 想法延伸：Ducking（有人說話時自動壓低音樂）

**概念**：用 `detect_nonsilent` 找出人聲段，在這些時間把 BGM 再多降 `duck_db`；沒人說話時音樂回到原本的音量。增益曲線要**平滑**變化（attack／release），否則音樂音量會忽大忽小地「抽動」。

做法：先在「每毫秒一點」的解析度建立增益曲線、用移動平均平滑，再內插到每個樣本，乘到 BGM 上。

> **先預測再執行**：`pad_ms`（在語音段前後多壓一點）設太小會發生什麼事？

In [ ]:
def duck_bgm(voice, bed, duck_db=10, min_silence_len=400, relative_thresh_db=14,
             pad_ms=150, smooth_ms=200):
    """回傳 (壓低後的 BGM, 增益曲線 dB, 語音段)。"""
    thr = get_relative_threshold(voice, relative_thresh_db)
    spans = [] if thr is None else detect_nonsilent(voice, min_silence_len=min_silence_len,
                                                    silence_thresh=thr)
    gain_ms = np.zeros(len(bed))                     # 每毫秒一個增益值（dB）
    for start, end in spans:
        gain_ms[max(0, start - pad_ms): end + pad_ms] = -duck_db
    kernel = np.ones(smooth_ms) / smooth_ms                 # 平滑：避免音樂忽大忽小
    padded = np.pad(gain_ms, (smooth_ms // 2, smooth_ms - smooth_ms // 2 - 1), mode="edge")
    gain_ms = np.convolve(padded, kernel, mode="valid")
    y_bed, sr = seg_to_np(bed)
    t_ms = np.arange(len(y_bed)) * 1000 / sr
    gain_db = np.interp(t_ms, np.arange(len(gain_ms)), gain_ms)
    return np_to_seg(y_bed * 10 ** (gain_db / 20), sr), gain_ms, spans


# 為了聽清楚 ducking 的「放開」，這裡改用保留 1.2 秒換氣停頓的版本當人聲
voice_breath = shift_average_dbfs(compact_long_silence(audio, keep_gap_ms=1200), -18.0)
bed = AudioSegment.silent(duration=len(voice_breath), frame_rate=voice_breath.frame_rate)
bed = bed.overlay(bgm - 18, loop=True)
plain_mix = bed.overlay(voice_breath)
ducked_bed, gain_curve, duck_spans = duck_bgm(voice_breath, bed, duck_db=10)
ducked_mix = ducked_bed.overlay(voice_breath)
voice = voice_breath

fig, axes = plt.subplots(2, 1, figsize=(11, 4.5), sharex=True)
yv, sr = seg_to_np(voice)
axes[0].plot(np.arange(len(yv)) / sr, yv, lw=0.4, color="0.3")
axes[0].set_title("Voice")
axes[0].set_ylabel("Amplitude")
axes[1].plot(np.arange(len(gain_curve)) / 1000, gain_curve, color="tab:red")
axes[1].set_title("Extra BGM gain while speaking (ducking)")
axes[1].set_ylabel("Gain (dB)")
axes[1].set_xlabel("Time (s)")
plt.tight_layout()
plt.show()

describe(bed, "BGM 原本")
describe(ducked_bed, "BGM ducking 後")
describe(plain_mix, "無 ducking")
describe(ducked_mix, "有 ducking")
play(plain_mix, "無 ducking")
play(ducked_mix, "有 ducking（說話時音樂再降 10 dB）")

**聆聽檢核**
- [ ] BGM 在整段人聲中都存在，沒有中途斷掉。
- [ ] 有 ducking 時人聲更清楚，停頓處音樂會稍微浮上來。
- [ ] 音樂音量的變化是平滑的，沒有「抽動」感。

> 想一想：如果人聲已經用 `keep_gap_ms=300` 壓縮過，停頓都短於 `min_silence_len=400`，ducking 會變成什麼樣子？（可以把 `voice_breath` 換成 `podcast` 試試看。）

## 11.20 高通與低通濾波：回到 W10 的頻譜

**概念**：
- **高通（high-pass）**：保留高於截止頻率的成分，常用來去除低頻嗡嗡聲、腳步、冷氣低頻。
- **低通（low-pass）**：保留低於截止頻率的成分，可削弱高頻嘶聲，但聲音也會變悶。

下面先在人聲上人工加入 **60 Hz 市電嗡聲**（臺灣市電頻率）與其 120 Hz 諧波，再加上高頻嘶聲，用 W10 的方法畫平均頻譜比較前後。pydub 的濾波器是一階（每八度約 6 dB），衰減比較溫和，所以也比較「串兩次」的效果。

> **先預測再執行**：`high_pass_filter(120)` 能把 60 Hz 嗡聲完全消掉嗎？

In [ ]:
excerpt = podcast[:8000]
y_ex, sr = seg_to_np(excerpt)
t = np.arange(len(y_ex)) / sr
hum = 0.05 * np.sin(2 * np.pi * 60 * t) + 0.02 * np.sin(2 * np.pi * 120 * t)
hiss = 0.01 * rng.standard_normal(len(t))
noisy = np_to_seg(y_ex * 0.8 + hum + hiss, sr)

hp1 = noisy.high_pass_filter(120)
hp2 = noisy.high_pass_filter(120).high_pass_filter(120)
lp = noisy.low_pass_filter(3000)


def band_db(seg, f_lo, f_hi):
    f, db = avg_spectrum_db(seg)
    return float(db[(f >= f_lo) & (f <= f_hi)].max())


rows = [(name, round(band_db(s, 55, 65), 1), round(band_db(s, 6000, 10000), 1))
        for name, s in [("noisy", noisy), ("high_pass 120", hp1),
                        ("high_pass 120 ×2", hp2), ("low_pass 3000", lp)]]
show_table(["版本", "60 Hz 附近峰值 (dB)", "6–10 kHz 峰值 (dB)"], rows)

plot_spectra([noisy, hp1, hp2], ["noisy", "high_pass_filter(120)", "high_pass_filter(120) x2"],
             "High-pass: removing 60 Hz hum")
plot_spectra([noisy, lp], ["noisy", "low_pass_filter(3000)"], "Low-pass: reducing hiss")

play(noisy, "加入嗡聲與嘶聲")
play(hp2, "高通 120 Hz ×2")
play(lp, "低通 3000 Hz")

**觀察**：截止頻率以下的曲線往下彎，但不是「一刀切」；一階濾波器在 60 Hz（截止頻率下一個八度）只衰減約 6–7 dB，串兩次約加倍。專業軟體常用更高階的濾波器。

**聆聽檢核**
- [ ] 高通後嗡聲變小，人聲幾乎不受影響。
- [ ] 低通後嘶聲變小，但人聲也變悶——濾波是取捨。
- [ ] 我能在頻譜圖上指出 60 Hz、120 Hz 的尖峰。

## 11.21 變速不變調：`time_stretch`

**概念**：`rate > 1` 變快（變短）、`rate < 1` 變慢（變長），音高保持不變。librosa 內部用的是**相位聲碼器（phase vocoder）**：

1. 對聲音做 **STFT**（W10）得到一格一格的頻譜。
2. 依 `rate` 重新安排影格的時間間隔，並修正每個頻率格的相位，讓相鄰影格接得起來。
3. 用 **ISTFT** 還原成波形。

頻率內容（音高）沒有被動到，只改了「影格之間的時間」。

> **先預測再執行**：8 秒的片段用 `rate=1.25` 與 `rate=0.8` 處理後，各是幾秒？

In [ ]:
y, sr = seg_to_np(podcast[:8000])

fast = librosa.effects.time_stretch(y, rate=1.25)
slow = librosa.effects.time_stretch(y, rate=0.8)
print(f"原始 {len(y) / sr:.2f} s | rate=1.25 → {len(fast) / sr:.2f} s | rate=0.8 → {len(slow) / sr:.2f} s")

# 直接看 phase vocoder：STFT 影格數變成 1/rate 倍
D = librosa.stft(y, n_fft=2048, hop_length=512)
D_fast = librosa.phase_vocoder(D, rate=1.25)
print("STFT 形狀（頻率格, 影格）:", D.shape, "→ phase_vocoder(rate=1.25):", D_fast.shape)

play(np_to_seg(y, sr), "原始")
play(np_to_seg(fast, sr), "time_stretch rate=1.25（變快，音高不變）")
play(np_to_seg(slow, sr), "time_stretch rate=0.8（變慢，音高不變）")

## 11.22 變調不變速：`pitch_shift`，對照「直接改播放速度」

**概念**：`n_steps` 以半音為單位，+12 是高八度；頻率倍數 = 2^(n_steps/12)，+4 半音 ≈ ×1.26。librosa 的作法是「先 time stretch，再重新取樣」，所以長度不變、音高改變。

對照組：**直接改播放取樣率**（像錄音帶轉快）——不做任何頻譜處理，長度和音高**同時**改變，就是「花栗鼠聲」。

> **先預測再執行**：用 1.25 倍速播放，長度變幾秒？音高升幾個半音？（提示：12·log2(1.25)）

In [ ]:
up = librosa.effects.pitch_shift(y, sr=sr, n_steps=4)
down = librosa.effects.pitch_shift(y, sr=sr, n_steps=-4)


def naive_speed(seg, factor):
    """直接改播放取樣率（不做頻譜處理），再轉回原取樣率以便比較。"""
    sped = seg._spawn(seg.raw_data, overrides={"frame_rate": int(seg.frame_rate * factor)})
    return sped.set_frame_rate(seg.frame_rate)


def median_f0(y, sr):
    """以 YIN 估計基頻，只取較大聲的影格（略過靜音）後取中位數。"""
    f0 = librosa.yin(y, fmin=65, fmax=500, sr=sr, frame_length=2048)
    rms = librosa.feature.rms(y=y, frame_length=2048, hop_length=512)[0]
    n = min(len(f0), len(rms))
    voiced = rms[:n] > np.percentile(rms[:n], 60)
    return float(np.median(f0[:n][voiced]))


chip, _ = seg_to_np(naive_speed(np_to_seg(y, sr), 1.25))
versions = [("原始", y), ("time_stretch 1.25", fast), ("pitch_shift +4", up),
            ("pitch_shift -4", down), ("直接 1.25 倍速", chip)]
f0_ref = median_f0(y, sr)
rows = []
for name, sig in versions:
    f0 = median_f0(sig, sr)
    rows.append((name, round(len(sig) / sr, 2), round(f0, 1), round(12 * np.log2(f0 / f0_ref), 2)))
show_table(["版本", "長度 (s)", "中位基頻 (Hz)", "相對原始（半音）"], rows)
print(f"理論值：+4 半音 = ×{2 ** (4 / 12):.3f}；1.25 倍速 = +{12 * np.log2(1.25):.2f} 半音")

fig, axes = plt.subplots(2, 2, figsize=(12, 6.5), sharey=True)
for ax, (name, sig) in zip(axes.ravel(), [versions[0], versions[1], versions[2], versions[4]]):
    S = librosa.amplitude_to_db(np.abs(librosa.stft(sig, n_fft=2048, hop_length=512)), ref=np.max)
    librosa.display.specshow(S, sr=sr, hop_length=512, x_axis="time", y_axis="log", ax=ax)
    ax.set_ylim(60, 4000)
en = {"原始": "original", "time_stretch 1.25": "time_stretch rate=1.25",
      "pitch_shift +4": "pitch_shift n_steps=+4", "直接 1.25 倍速": "naive playback x1.25"}
for ax, (name, _) in zip(axes.ravel(), [versions[0], versions[1], versions[2], versions[4]]):
    ax.set_title(en[name])
plt.tight_layout()
plt.show()

play(np_to_seg(up, sr), "pitch_shift +4（音高升、長度不變）")
play(np_to_seg(down, sr), "pitch_shift -4（音高降、長度不變）")
play(np_to_seg(chip, sr), "直接 1.25 倍速（長度與音高一起變）")

**觀察**：頻譜圖上，time stretch 只在**時間軸**被壓縮；pitch shift 只在**頻率軸**往上移；直接倍速則兩個軸同時改變。

**聆聽檢核**
- [ ] time stretch 後說話速度改變，但聲音不變尖或變低沉。
- [ ] pitch shift 後長度不變，但聲音變尖／變沉；仔細聽，可能有些「金屬感」或「迴音感」（phase vocoder 的副作用）。
- [ ] 直接倍速的版本聽起來像花栗鼠——長度和音高被綁在一起。

## 延伸練習：參數比較

1. time stretch：rate = 0.8／1.0／1.25／1.5
2. pitch shift：n_steps = −4／0／+4
3. BGM 音量：−12／−18／−24 dB（「人聲比 BGM 大多少」= voice dBFS − BGM dBFS）

執行後填表，並寫下：**數字正確（長度、基頻都符合理論）是否就代表聽起來自然？**

| 參數 | 長度 (s) | 中位基頻 (Hz) 或 人聲−BGM (dB) | 聽感 |
|---|---|---|---|
| rate 0.8 | | | |
| rate 1.0 | | | |
| rate 1.25 | | | |
| rate 1.5 | | | |
| n_steps −4 | | | |
| n_steps 0 | | | |
| n_steps +4 | | | |
| BGM −12 dB | — | | |
| BGM −18 dB | — | | |
| BGM −24 dB | — | | |

In [ ]:
ext_rows = []
for rate in [0.8, 1.0, 1.25, 1.5]:          # TODO（選做）：加入其他倍率
    out = librosa.effects.time_stretch(y, rate=rate)
    ext_rows.append((f"rate {rate}", round(len(out) / sr, 2), round(median_f0(out, sr), 1)))
    play(np_to_seg(out, sr), f"time_stretch rate={rate}")
for steps in [-4, 0, 4]:
    out = librosa.effects.pitch_shift(y, sr=sr, n_steps=steps)
    ext_rows.append((f"n_steps {steps:+d}", round(len(out) / sr, 2), round(median_f0(out, sr), 1)))
    play(np_to_seg(out, sr), f"pitch_shift n_steps={steps:+d}")
show_table(["參數", "長度 (s)", "中位基頻 (Hz)"], ext_rows)

bgm_rows = []
voice10 = podcast[:10000]
for g in [-12, -18, -24]:
    bed = AudioSegment.silent(duration=len(voice10), frame_rate=voice10.frame_rate).overlay(bgm + g, loop=True)
    bgm_rows.append((f"BGM {g} dB", round(voice10.dBFS - bed.dBFS, 1), count_clipped(bed.overlay(voice10))))
    play(bed.overlay(voice10), f"BGM {g} dB")
show_table(["BGM 增益", "人聲 − BGM (dB)", "滿刻度樣本"], bgm_rows)

## 檢核題

請用自己的話回答（每題 2–4 句），可以引用本 notebook 的數字或圖。

1. 為什麼 `audio[5:15]` 拿不到第 5 到第 15 秒？正確寫法是什麼？
2. `audio + 6` 和 `audio + audio2` 都用 `+`，差別在哪？輸出長度各如何變化？
3. 串接（`+`／`append`）與疊加（`overlay`）有什麼不同？疊加時為什麼要留 headroom？
4. 為什麼全靜音的 `dBFS` 是 −∞？程式中要怎麼防範？
5. 為什麼錄音中的長靜音會讓「平均音量調整」失真？Podcast Editor 的步驟順序如何避免？
6. 為什麼壓縮停頓時不把停頓全部刪掉（`keep_gap_ms=0`）？
7. 相位聲碼器（phase vocoder）和 W10 的 STFT 有什麼關係？time stretch 改變的是頻譜圖的哪一個軸？
8. 長度、dBFS、基頻都符合理論值，為什麼聽起來仍可能不自然？請舉一個你在本週聽到的例子。

### 我的答案

1.
2.
3.
4.
5.
6.
7.
8.

## 繳交清單

- [ ] 本 notebook（`.ipynb`），已執行完成並保留輸出
- [ ] 練習 1 表格：處理前後的長度、dBFS、max dBFS、波形圖與聆聽描述
- [ ] 練習 2 表格：Podcast 自動剪輯參數與結果、**至少 3 條聆聽筆記**；附 `my_podcast.mp3`（建議使用自己的 1–3 分鐘錄音）
- [ ] 練習 3 表格：`min_silence_len` 300／600／900／1200 ms 的語音段數與輸出時長，並說明你選擇哪一個值與理由
- [ ] （延伸）time stretch／pitch shift／BGM 音量比較表
- [ ] 檢核題 8 題作答

> 下載檔案：在 Colab 左側「檔案」面板對 `my_podcast.mp3` 按右鍵 → 下載。